# 🎯 Phase 8: Milestone Exam Solutions

> **SQL Mastery & Database Architecture**
>
> This notebook contains comprehensive solutions for all five Phase Milestone Exam questions.
> SQL queries use Python's `sqlite3` module for Pyodide compatibility.
> PostgreSQL-specific features are explained in markdown cells.

---

## Question 1: Advanced Recursive CTE — Organization Chart

**Combines**: Recursive CTEs (Day 85), Hierarchical Data (Day 87)

**Scenario**: Build a recursive query that traverses an employee hierarchy to:
1. Calculate management depth for each employee
2. Compute total team size (direct + indirect reports)
3. Generate the reporting chain (path from CEO to employee)

### Theory: Recursive CTEs

A recursive CTE has two parts:
1. **Anchor**: The base case (e.g., the CEO with no manager)
2. **Recursive**: Joins the CTE to itself to traverse the hierarchy

```sql
WITH RECURSIVE cte AS (
    -- Anchor: starting rows
    SELECT id, name, manager_id, 0 AS depth
    FROM employees WHERE manager_id IS NULL
    
    UNION ALL
    
    -- Recursive: join children to parents
    SELECT e.id, e.name, e.manager_id, cte.depth + 1
    FROM employees e
    JOIN cte ON e.manager_id = cte.id
)
SELECT * FROM cte;
```

⚠️ **Safety**: Always add a depth limit to prevent infinite loops!

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.executescript("""
    CREATE TABLE employees (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        title TEXT NOT NULL,
        department TEXT,
        manager_id INTEGER REFERENCES employees(id),
        salary REAL
    );

    INSERT INTO employees VALUES
        (1, 'Alice Chen', 'CEO', 'Executive', NULL, 250000),
        (2, 'Bob Smith', 'CTO', 'Engineering', 1, 200000),
        (3, 'Carol Jones', 'CFO', 'Finance', 1, 195000),
        (4, 'David Kim', 'VP Engineering', 'Engineering', 2, 175000),
        (5, 'Elena Garcia', 'VP Data', 'Engineering', 2, 170000),
        (6, 'Frank Wilson', 'VP Finance', 'Finance', 3, 160000),
        (7, 'Grace Lee', 'Staff Engineer', 'Engineering', 4, 150000),
        (8, 'Henry Brown', 'Sr Engineer', 'Engineering', 4, 140000),
        (9, 'Irene Davis', 'Data Scientist', 'Engineering', 5, 135000),
        (10, 'James Martinez', 'Data Engineer', 'Engineering', 5, 130000),
        (11, 'Karen White', 'Sr Accountant', 'Finance', 6, 110000),
        (12, 'Liam Johnson', 'Jr Engineer', 'Engineering', 7, 95000),
        (13, 'Maya Patel', 'Jr Engineer', 'Engineering', 7, 92000),
        (14, 'Nathan Lee', 'Analyst', 'Finance', 6, 85000),
        (15, 'Olivia Clark', 'Intern', 'Engineering', 8, 60000);
""")
conn.commit()
print("✅ Organization data loaded (15 employees)")

In [ ]:
# Recursive CTE: Org chart with depth and reporting chain
print("=" * 65)
print("ORG CHART — RECURSIVE CTE")
print("=" * 65)

query = """
    WITH RECURSIVE org_tree AS (
        -- Anchor: CEO (no manager)
        SELECT
            id, name, title, manager_id,
            0 AS depth,
            name AS chain
        FROM employees
        WHERE manager_id IS NULL

        UNION ALL

        -- Recursive: each employee under their manager
        SELECT
            e.id, e.name, e.title, e.manager_id,
            o.depth + 1,
            o.chain || ' → ' || e.name
        FROM employees e
        JOIN org_tree o ON e.manager_id = o.id
        WHERE o.depth < 10  -- Safety: prevent infinite loops
    )
    SELECT
        id, depth,
        SUBSTR('                    ', 1, depth * 4) || name AS indented_name,
        title, chain
    FROM org_tree
    ORDER BY chain;
"""

rows = cur.execute(query).fetchall()
print(f"\n{'ID':>4s} {'Depth':>5s}  {'Name':40s} {'Title'}")
print("-" * 80)
for id_, depth, name, title, chain in rows:
    print(f"{id_:>4d} {'│' * depth:>5s}  {name:40s} {title}")

In [ ]:
# Team size calculation (direct + indirect reports)
print("\n" + "=" * 65)
print("TEAM SIZE (Direct + Indirect Reports)")
print("=" * 65)

query = """
    WITH RECURSIVE subordinates AS (
        SELECT id, name, manager_id, id AS root_manager
        FROM employees

        UNION ALL

        SELECT e.id, e.name, e.manager_id, s.root_manager
        FROM employees e
        JOIN subordinates s ON e.manager_id = s.id
        WHERE s.id != s.root_manager OR s.manager_id IS NOT NULL
    )
    SELECT
        m.name AS manager,
        m.title,
        COUNT(DISTINCT s.id) - 1 AS team_size  -- Exclude self
    FROM employees m
    LEFT JOIN subordinates s ON s.root_manager = m.id AND s.id != m.id
    GROUP BY m.id, m.name, m.title
    HAVING team_size > 0
    ORDER BY team_size DESC;
"""

# Alternative simpler approach using the recursive CTE
query2 = """
    WITH RECURSIVE tree AS (
        SELECT id AS root, id AS descendant FROM employees
        UNION ALL
        SELECT t.root, e.id
        FROM tree t
        JOIN employees e ON e.manager_id = t.descendant
    )
    SELECT
        e.name,
        e.title,
        COUNT(t.descendant) - 1 AS total_reports
    FROM employees e
    JOIN tree t ON t.root = e.id
    GROUP BY e.id, e.name, e.title
    HAVING total_reports > 0
    ORDER BY total_reports DESC;
"""

rows = cur.execute(query2).fetchall()
print(f"\n{'Manager':>20s} {'Title':>20s} {'Total Reports':>15s}")
print("-" * 58)
for name, title, reports in rows:
    bar = "█" * reports
    print(f"{name:>20s} {title:>20s} {reports:>15d} {bar}")

---

## Question 2: Database Indexing & Performance

**Combines**: Indexing (Day 86), Query Optimization (Day 88)

**Scenario**: Analyze query performance and design indexes for common query patterns.

### Theory: Index Types

| Index Type | Use Case | Trade-off |
|------------|----------|----------|
| **B-Tree** (default) | Equality (`=`) and range (`<`, `>`, `BETWEEN`) | Good for most queries |
| **Hash** | Equality only (`=`) | Faster for exact lookups, no range |
| **GIN** | Full-text search, JSONB, arrays | Multi-valued columns |
| **GiST** | Geometric, spatial data | Nearest-neighbor, containment |
| **BRIN** | Large sorted tables (time-series) | Tiny index, good for sequential data |

### SARGable Queries

**SARGable** (Search ARGument able): queries that can use indexes.

```sql
-- ✅ SARGable (can use index on sale_date):
WHERE sale_date >= '2024-01-01'

-- ❌ NOT SARGable (function prevents index use):
WHERE YEAR(sale_date) = 2024
WHERE UPPER(name) = 'ALICE'
WHERE price + tax > 100
```

### PostgreSQL-Specific: Partial and Expression Indexes

```sql
-- Partial index: only index active users (smaller, faster)
CREATE INDEX idx_active_users ON users(email) WHERE status = 'active';

-- Expression index: index computed value
CREATE INDEX idx_lower_email ON users(LOWER(email));

-- Covering index (INCLUDE): avoid table lookup
CREATE INDEX idx_orders_covering
    ON orders(customer_id) INCLUDE (total, status);
```

In [ ]:
# Demonstrate index creation and EXPLAIN
import random

print("=" * 60)
print("INDEX DESIGN EXERCISE")
print("=" * 60)

# Create a larger table for realistic indexing demo
cur.executescript("""
    CREATE TABLE orders (
        id INTEGER PRIMARY KEY,
        customer_id INTEGER,
        order_date TEXT,
        status TEXT CHECK(status IN ('pending','shipped','delivered','cancelled')),
        total REAL,
        product_category TEXT
    );
""")

# Insert 1000 rows
random.seed(42)
statuses = ["pending", "shipped", "delivered", "cancelled"]
categories = ["Electronics", "Books", "Clothing", "Home", "Food"]
for i in range(1, 1001):
    cur.execute(
        "INSERT INTO orders VALUES (?,?,?,?,?,?)",
        (
            i,
            random.randint(1, 200),
            f"2024-{random.randint(1, 12):02d}-{random.randint(1, 28):02d}",
            random.choice(statuses),
            round(random.uniform(10, 500), 2),
            random.choice(categories),
        ),
    )
conn.commit()

# Common query patterns and recommended indexes
queries = [
    {
        "name": "Orders by customer",
        "query": "SELECT * FROM orders WHERE customer_id = 42",
        "index": "CREATE INDEX idx_orders_customer ON orders(customer_id)",
        "reason": "Equality lookup on customer_id — B-Tree index",
    },
    {
        "name": "Date range query",
        "query": "SELECT * FROM orders WHERE order_date BETWEEN '2024-03-01' AND '2024-03-31'",
        "index": "CREATE INDEX idx_orders_date ON orders(order_date)",
        "reason": "Range scan on order_date — B-Tree index",
    },
    {
        "name": "Status + date compound",
        "query": "SELECT * FROM orders WHERE status = 'pending' AND order_date > '2024-06-01'",
        "index": "CREATE INDEX idx_orders_status_date ON orders(status, order_date)",
        "reason": "Compound index: equality first (status), then range (date)",
    },
]

for q in queries:
    print(f"\n📋 Pattern: {q['name']}")
    print(f"   Query:  {q['query']}")

    # Show EXPLAIN before index
    plan_before = cur.execute(f"EXPLAIN QUERY PLAN {q['query']}").fetchall()
    print(f"   Before: {plan_before[0][-1]}")

    # Create index
    cur.execute(q["index"])

    # Show EXPLAIN after index
    plan_after = cur.execute(f"EXPLAIN QUERY PLAN {q['query']}").fetchall()
    print(f"   After:  {plan_after[0][-1]}")
    print(f"   Index:  {q['index']}")
    print(f"   Why:    {q['reason']}")

---

## Question 3: Transaction Isolation & Concurrency

**Combines**: Transactions (Day 89), ACID Properties (Day 87)

**Scenario**: Explain and demonstrate the four transaction isolation levels and the anomalies they prevent.

### Theory: ACID Properties

| Property | Meaning | Example |
|----------|---------|--------|
| **Atomicity** | All or nothing | Transfer: debit AND credit both succeed or both roll back |
| **Consistency** | Valid state → valid state | Balance never goes negative (if constrained) |
| **Isolation** | Concurrent txns don't interfere | Two users reading same row see consistent data |
| **Durability** | Committed = permanent | Server crash after commit → data persists |

### Isolation Levels

| Level | Dirty Read | Non-Repeatable Read | Phantom Read | Performance |
|-------|-----------|--------------------|--------------|-----------|
| READ UNCOMMITTED | ⚠️ Possible | ⚠️ Possible | ⚠️ Possible | Fastest |
| READ COMMITTED | ✅ Prevented | ⚠️ Possible | ⚠️ Possible | Good |
| REPEATABLE READ | ✅ Prevented | ✅ Prevented | ⚠️ Possible | Slower |
| SERIALIZABLE | ✅ Prevented | ✅ Prevented | ✅ Prevented | Slowest |

### Anomaly Definitions

- **Dirty Read**: Reading data that another transaction hasn't yet committed
- **Non-Repeatable Read**: Same query returns different values within one transaction
- **Phantom Read**: Same query returns different set of rows within one transaction

In [ ]:
# Demonstration: Bank Transfer with ACID guarantees


def create_bank_db():
    conn = sqlite3.connect(":memory:")
    cur = conn.cursor()
    cur.executescript("""
        CREATE TABLE accounts (
            id INTEGER PRIMARY KEY,
            name TEXT,
            balance REAL CHECK(balance >= 0)
        );
        INSERT INTO accounts VALUES (1, 'Alice', 1000.00);
        INSERT INTO accounts VALUES (2, 'Bob', 500.00);
        INSERT INTO accounts VALUES (3, 'Carol', 250.00);
    """)
    conn.commit()
    return conn


def transfer(conn, from_id, to_id, amount):
    """
    Atomic bank transfer with proper error handling.

    If any step fails, the entire transfer rolls back.
    """
    cur = conn.cursor()
    try:
        # Check balance
        balance = cur.execute(
            "SELECT balance FROM accounts WHERE id = ?", (from_id,)
        ).fetchone()[0]

        if balance < amount:
            raise ValueError(f"Insufficient funds: ${balance:.2f} < ${amount:.2f}")

        # Debit sender
        cur.execute(
            "UPDATE accounts SET balance = balance - ? WHERE id = ?", (amount, from_id)
        )

        # Credit receiver
        cur.execute(
            "UPDATE accounts SET balance = balance + ? WHERE id = ?", (amount, to_id)
        )

        conn.commit()
        return True, f"Transferred ${amount:.2f}"

    except Exception as e:
        conn.rollback()
        return False, str(e)


def show_balances(conn, label=""):
    rows = conn.execute("SELECT name, balance FROM accounts ORDER BY id").fetchall()
    total = sum(b for _, b in rows)
    print(
        f"  {label}: "
        + " | ".join(f"{n}: ${b:.2f}" for n, b in rows)
        + f" | Total: ${total:.2f}"
    )


# Demo
print("=" * 60)
print("ACID TRANSACTION DEMO")
print("=" * 60)

bank = create_bank_db()
show_balances(bank, "Initial")

# Successful transfer
ok, msg = transfer(bank, 1, 2, 300)
print(f"\n  Transfer Alice→Bob $300: {'✅' if ok else '❌'} {msg}")
show_balances(bank, "After  ")

# Failed transfer (insufficient funds)
ok, msg = transfer(bank, 3, 1, 500)
print(f"\n  Transfer Carol→Alice $500: {'✅' if ok else '❌'} {msg}")
show_balances(bank, "After  ")

print("\n💡 Note: Total balance is always $1,750 (conservation of money).")
print("   The failed transfer was rolled back — ATOMICITY in action.")

bank.close()

---

## Question 4: Stored Procedures & Triggers

**Combines**: Stored Procedures (Day 90), Security (Day 91)

**Scenario**: Implement database-side logic for:
1. Audit logging trigger (tracks all changes)
2. Stored procedure for safe data modification

### Theory: Triggers vs Stored Procedures

| Feature | Trigger | Stored Procedure |
|---------|---------|------------------|
| **When** | Automatic (on INSERT/UPDATE/DELETE) | Manual (CALL/EXECUTE) |
| **Purpose** | Enforce rules, audit logging | Business logic, batch operations |
| **Visibility** | Hidden (can surprise developers) | Explicit (called by name) |
| **Best For** | Auditing, computed columns | Complex transactions, reporting |

> ⚠️ SQLite supports triggers but not full stored procedures. We demonstrate triggers here and show PostgreSQL procedure syntax in markdown.

In [ ]:
# Audit logging with triggers
conn3 = sqlite3.connect(":memory:")
cur3 = conn3.cursor()

cur3.executescript("""
    CREATE TABLE products (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        price REAL NOT NULL,
        stock INTEGER DEFAULT 0
    );

    CREATE TABLE audit_log (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        table_name TEXT,
        action TEXT,
        record_id INTEGER,
        old_values TEXT,
        new_values TEXT,
        changed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );

    -- Trigger: Log INSERT
    CREATE TRIGGER log_product_insert
    AFTER INSERT ON products
    BEGIN
        INSERT INTO audit_log (table_name, action, record_id, new_values)
        VALUES ('products', 'INSERT', NEW.id,
            'name=' || NEW.name || ', price=' || NEW.price || ', stock=' || NEW.stock);
    END;

    -- Trigger: Log UPDATE
    CREATE TRIGGER log_product_update
    AFTER UPDATE ON products
    BEGIN
        INSERT INTO audit_log (table_name, action, record_id, old_values, new_values)
        VALUES ('products', 'UPDATE', OLD.id,
            'name=' || OLD.name || ', price=' || OLD.price || ', stock=' || OLD.stock,
            'name=' || NEW.name || ', price=' || NEW.price || ', stock=' || NEW.stock);
    END;

    -- Trigger: Log DELETE
    CREATE TRIGGER log_product_delete
    AFTER DELETE ON products
    BEGIN
        INSERT INTO audit_log (table_name, action, record_id, old_values)
        VALUES ('products', 'DELETE', OLD.id,
            'name=' || OLD.name || ', price=' || OLD.price || ', stock=' || OLD.stock);
    END;

    -- Trigger: Prevent negative stock
    CREATE TRIGGER prevent_negative_stock
    BEFORE UPDATE ON products
    WHEN NEW.stock < 0
    BEGIN
        SELECT RAISE(ABORT, 'Stock cannot be negative');
    END;
""")
conn3.commit()
print("✅ Triggers created (INSERT, UPDATE, DELETE audit + stock guard)")

In [ ]:
# Demonstrate triggers
print("=" * 60)
print("TRIGGER AUDIT LOG DEMO")
print("=" * 60)

# INSERT
cur3.execute("INSERT INTO products VALUES (1, 'Widget', 29.99, 100)")
cur3.execute("INSERT INTO products VALUES (2, 'Gadget', 49.99, 50)")
print("\n1. Inserted Widget and Gadget")

# UPDATE
cur3.execute("UPDATE products SET price = 34.99 WHERE id = 1")
print("2. Updated Widget price: $29.99 → $34.99")

cur3.execute("UPDATE products SET stock = stock - 10 WHERE id = 2")
print("3. Reduced Gadget stock by 10")

# DELETE
cur3.execute("DELETE FROM products WHERE id = 2")
print("4. Deleted Gadget")

# Try negative stock (should fail)
print("5. Attempting to set negative stock...")
try:
    cur3.execute("UPDATE products SET stock = -5 WHERE id = 1")
    print("   ❌ Should have failed!")
except sqlite3.IntegrityError as e:
    print(f"   ✅ Blocked by trigger: {e}")

conn3.commit()

# Show audit log
print(f"\n{'─' * 60}")
print("AUDIT LOG")
print(f"{'─' * 60}")

logs = cur3.execute("""
    SELECT action, record_id, old_values, new_values, changed_at
    FROM audit_log ORDER BY id
""").fetchall()

for action, rid, old, new, ts in logs:
    icon = {"INSERT": "🟢", "UPDATE": "🔄", "DELETE": "🔴"}[action]
    print(f"  {icon} {action:7s} record #{rid}")
    if old:
        print(f"    Old: {old}")
    if new:
        print(f"    New: {new}")

conn3.close()

### PostgreSQL Stored Procedure Example

```sql
-- Stored procedure for safe inventory adjustment
CREATE OR REPLACE PROCEDURE adjust_inventory(
    p_product_id INT,
    p_quantity INT,
    p_reason TEXT
)
LANGUAGE plpgsql AS $$
DECLARE
    v_current_stock INT;
BEGIN
    -- Lock the row to prevent concurrent modifications
    SELECT stock INTO v_current_stock
    FROM products WHERE id = p_product_id
    FOR UPDATE;  -- Row-level lock

    IF v_current_stock IS NULL THEN
        RAISE EXCEPTION 'Product % not found', p_product_id;
    END IF;

    IF v_current_stock + p_quantity < 0 THEN
        RAISE EXCEPTION 'Insufficient stock: % + % < 0', v_current_stock, p_quantity;
    END IF;

    -- Update stock
    UPDATE products SET stock = stock + p_quantity WHERE id = p_product_id;

    -- Log the adjustment
    INSERT INTO inventory_adjustments (product_id, quantity, reason, adjusted_at)
    VALUES (p_product_id, p_quantity, p_reason, NOW());

    COMMIT;
END;
$$;

-- Usage:
CALL adjust_inventory(1, -10, 'Sold to customer #42');
CALL adjust_inventory(1, 50, 'Restocked from warehouse');
```

---

## Question 5: Query Performance Tuning

**Combines**: Performance Tuning (Day 92), Partitioning (Day 86)

**Scenario**: Analyze and optimize slow queries.

### Performance Anti-Patterns & Fixes

| Anti-Pattern | Why It's Slow | Fix |
|-------------|--------------|-----|
| `SELECT *` | Reads all columns, waste I/O | Select only needed columns |
| `WHERE UPPER(col)` | Function prevents index use | Expression index or store normalized |
| `NOT IN (subquery)` | Can't use index efficiently | Use `NOT EXISTS` or `LEFT JOIN ... IS NULL` |
| `LIKE '%search%'` | Leading wildcard = full scan | Full-text search (GIN index) |
| Correlated subquery | Executes once per outer row | Rewrite as JOIN |
| Missing JOIN index | Hash join on large tables | Index the FK column |

### PostgreSQL EXPLAIN Output

```sql
EXPLAIN ANALYZE SELECT * FROM orders WHERE customer_id = 42;

-- Read this bottom-up:
-- Seq Scan on orders  (cost=0.00..245.00 rows=5 width=72)  ← Without index
-- Index Scan using idx_customer on orders  (cost=0.29..8.31 rows=5 width=72)  ← With index
```

Key metrics in EXPLAIN:
- **cost**: Estimated work (lower = better)
- **rows**: Estimated rows returned
- **actual time**: Real execution time (with ANALYZE)
- **Seq Scan** → Full table scan (bad for large tables)
- **Index Scan** → Using index (good)

In [ ]:
# Query optimization demo
print("=" * 60)
print("QUERY OPTIMIZATION EXAMPLES")
print("=" * 60)

optimizations = [
    {
        "name": "SELECT * → Specific columns",
        "slow": "SELECT * FROM orders WHERE customer_id = 42",
        "fast": "SELECT id, order_date, total FROM orders WHERE customer_id = 42",
        "why": "Reduces I/O by reading only needed columns. With covering index, avoids table lookup entirely.",
    },
    {
        "name": "NOT IN → NOT EXISTS",
        "slow": """SELECT * FROM employees WHERE id NOT IN
    (SELECT DISTINCT manager_id FROM employees WHERE manager_id IS NOT NULL)""",
        "fast": """SELECT e.* FROM employees e
    WHERE NOT EXISTS (SELECT 1 FROM employees m WHERE m.manager_id = e.id)""",
        "why": "NOT EXISTS short-circuits (stops at first match). NOT IN must scan entire subquery and handles NULL incorrectly.",
    },
    {
        "name": "Correlated subquery → JOIN",
        "slow": """SELECT name, (SELECT COUNT(*) FROM orders o WHERE o.customer_id = e.id) AS order_count
    FROM employees e""",
        "fast": """SELECT e.name, COUNT(o.id) AS order_count
    FROM employees e LEFT JOIN orders o ON o.customer_id = e.id
    GROUP BY e.id, e.name""",
        "why": "Correlated subquery runs N times (once per employee). JOIN runs once with hash/merge join.",
    },
]

for i, opt in enumerate(optimizations, 1):
    print(f"\n{'─' * 60}")
    print(f"Optimization #{i}: {opt['name']}")
    print(f"{'─' * 60}")
    print(f"  ❌ Slow: {opt['slow'][:80]}")
    print(f"  ✅ Fast: {opt['fast'][:80]}")
    print(f"  📝 Why:  {opt['why']}")

---

## 🎓 Summary

This notebook demonstrated solutions to all five Phase 8 Milestone Exam questions:

1. **Recursive CTEs**: Org chart traversal with depth, reporting chains, and team sizes
2. **Index Design**: B-Tree, partial/expression indexes, SARGable queries, EXPLAIN analysis
3. **ACID & Isolation**: Transaction atomicity demo, isolation level comparison
4. **Triggers & Procedures**: Audit logging triggers, stock guard, PostgreSQL stored procedures
5. **Query Optimization**: Anti-patterns (SELECT *, NOT IN, correlated subqueries) with proven fixes

Key takeaway: **Database mastery = understanding the optimizer**. If you know how indexes work and how the query planner thinks, you can make any query fast.